In [1]:
# !pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu128
!pip -q install unsloth
!pip -q install transformers==4.56.2
!pip -q install --no-deps trl==0.22.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 MB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 95.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 83.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 66.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 119.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6

In [2]:
import unsloth  # Important: import Unsloth early

import time
import torch

from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
assert torch.cuda.is_available(), "Please enable GPU runtime"
print("GPU:", torch.cuda.get_device_name(0))

GPU: Tesla T4


In [4]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
DATASET_SIZE = 200
MAX_SEQ_LENGTH = 1024
MAX_STEPS = 30
BATCH_SIZE = 2
GRAD_ACCUM = 4
LR = 2e-4

In [5]:
raw_dataset = load_dataset("yahma/alpaca-cleaned", split="train").select(range(DATASET_SIZE))

alpaca_prompt = """Below is an instruction.

### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}
"""

def convert_to_text(example):
    return {
        "text": alpaca_prompt.format(
            instruction=example["instruction"],
            input=example["input"],
            output=example["output"],
        )
    }

dataset = raw_dataset.map(
    convert_to_text,
    remove_columns=raw_dataset.column_names,
)

print("Sample training text:\n")
print(dataset[0]["text"])


README.md: 0.00B [00:00, ?B/s]

alpaca_data_cleaned.json:   0%|          | 0.00/44.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/51760 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Sample training text:

Below is an instruction.

### Instruction:
Give three tips for staying healthy.

### Input:


### Response:
1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.

2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.

3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.



In [6]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"


==((====))==  Unsloth 2026.6.7: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/762M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/438 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [7]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

model.print_trainable_parameters()


Not an error, but Unsloth cannot patch MLP layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2026.6.7 patched 22 layers with 22 QKV layers, 22 O layers and 0 MLP layers.


trainable params: 4,505,600 || all params: 1,104,553,984 || trainable%: 0.4079


In [8]:
sft_config = SFTConfig(
    max_steps=MAX_STEPS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,

    output_dir="unsloth_out",
    report_to="none",

    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,
    packing=False,

    # Keep same as HF code for fair classroom comparison
    fp16=False,
    bf16=False,

    optim="paged_adamw_8bit",
    logging_steps=5,
    save_strategy="no",
)


In [9]:
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=sft_config,
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/200 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [10]:
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()

start_time = time.time()

trainer.train()

torch.cuda.synchronize()

train_time = round(time.time() - start_time, 2)
train_vram_allocated = round(torch.cuda.max_memory_allocated() / 1024**3, 3)
train_vram_reserved = round(torch.cuda.max_memory_reserved() / 1024**3, 3)

print("\nUNSLOTH TRAINING RESULTS")
print("Train time/sec:", train_time)
print("Peak allocated VRAM/GB:", train_vram_allocated)
print("Peak reserved VRAM/GB:", train_vram_reserved)

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 200 | Num Epochs = 2 | Total steps = 30
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 4,505,600 of 1,104,553,984 (0.41% trained)


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
5,1.519900
10,1.588900
15,1.504700
20,1.427400
25,1.229900
30,1.352800



UNSLOTH TRAINING RESULTS
Train time/sec: 61.27
Peak allocated VRAM/GB: 0.931
Peak reserved VRAM/GB: 1.693


In [11]:
FastLanguageModel.for_inference(model)

instruction = "Give three tips for staying healthy."

inference_prompt = """Below is an instruction.

### Instruction:
{instruction}

### Input:


### Response:
""".format(instruction=instruction)

inputs = tokenizer(inference_prompt, return_tensors="pt").to("cuda")

# Warm-up generation
with torch.inference_mode():
    _ = model.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

torch.cuda.synchronize()
t0 = time.time()

with torch.inference_mode():
    output = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

torch.cuda.synchronize()
t1 = time.time()

input_tokens = inputs["input_ids"].shape[-1]
generated_token_ids = output[0][input_tokens:]

generated_tokens = len(generated_token_ids)
tokens_per_sec = round(generated_tokens / (t1 - t0), 2)

answer = tokenizer.decode(generated_token_ids, skip_special_tokens=True)

print("\nUNSLOTH FINAL RESULTS")
print("Train time/sec:", train_time)
print("Peak allocated VRAM/GB:", train_vram_allocated)
print("Peak reserved VRAM/GB:", train_vram_reserved)
print("Generated tokens/sec:", tokens_per_sec)

print("\nMODEL ANSWER:")
print(answer)


UNSLOTH FINAL RESULTS
Train time/sec: 61.27
Peak allocated VRAM/GB: 0.931
Peak reserved VRAM/GB: 1.693
Generated tokens/sec: 9.65

MODEL ANSWER:
1. Get enough sleep: Aim for 7-9 hours of sleep per night.

2. Eat a balanced diet: Include a variety of fruits, vegetables, whole grains, lean protein, and healthy fats in your diet.

3. Exercise regularly: Incorporate physical activity into your daily routine, such as walking, jogging, or swimming.

Remember, staying healthy is a lifelong journey, so be patient with yourself and keep trying new things.


In [ ]:
# Same base model
# Same dataset
# Same dataset size
# Same prompt format
# Same max steps
# Same batch size
# Same gradient accumulation
# Same learning rate
# Same LoRA r
# Same LoRA alpha
# Same LoRA target modules
# Same LoRA dropout
# Same max sequence length
# Same optimizer
# Same inference prompt
# Same max new tokens
# Same do_sample=False
# Same metric calculation

| Parameter                  | Meaning                                                    |        HF Result |  Unsloth Result | Better           | Simple Conclusion                                 |
| -------------------------- | ---------------------------------------------------------- | ---------------: | --------------: | ---------------- | ------------------------------------------------- |
| **Train time/sec**         | Time taken to complete fine-tuning                         |        94.46 sec |       61.27 sec | Lower is better  | **Unsloth trained ~1.54x faster**                 |
| **Peak allocated VRAM/GB** | Actual GPU memory used by tensors during training          |         1.717 GB |        0.931 GB | Lower is better  | **Unsloth used less actual GPU memory**           |
| **Peak reserved VRAM/GB**  | GPU memory reserved by PyTorch/CUDA for current/future use |         2.186 GB |        1.693 GB | Lower is better  | **Unsloth used less overall reserved GPU memory** |
| **Generated tokens/sec**   | Number of new tokens generated per second during inference | 12.61 tokens/sec | 9.65 tokens/sec | Higher is better | **HF inference was faster in this run**           |
